<a href="https://colab.research.google.com/github/Him-an-shi/Seminar/blob/main/Experiment_01_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# EXPERIMENT 1 — Environment Check
# This cell does not download any dataset.

import sys
import os
import platform
import shutil

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

# Check available disk space in the current environment
total, used, free = shutil.disk_usage("/content")
print(f"Free notebook disk space: {free / (1024**3):.2f} GB")

# Check PyTorch
try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Selected device:", device)
except ImportError:
    print("PyTorch is not installed.")

# Check basic numerical libraries
try:
    import numpy
    print("NumPy version:", numpy.__version__)
except ImportError:
    print("NumPy is not installed.")


Python version: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Free notebook disk space: 87.21 GB
PyTorch version: 2.11.0+cpu
CUDA available: False
Selected device: cpu
NumPy version: 2.1.3


In [ ]:

# EXPERIMENT 1: Single-task training smoke test
# Small built-in image dataset; no large downloads.

import sys
import time
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
except ImportError as e:
    print("Missing package:", e)
    print("Run this in a separate cell: %pip install torch scikit-learn")
    raise

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

# Load 8x8 images (no dataset download)
data = load_digits()
X = data.images.astype(np.float32) / 16.0
y = data.target.astype(np.int64)

# Fixed train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Convert images to tensors
X_train = torch.tensor(X_train).reshape(-1, 64)
X_test = torch.tensor(X_test).reshape(-1, 64)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)

# Small neural network
class SmallClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)

model = SmallClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train on one task
EPOCHS = 8
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader.dataset):.4f}"
    )

# Evaluate on held-out test data
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        predictions = outputs.argmax(dim=1)
        correct += (predictions.cpu() == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
elapsed = time.time() - start

print("\n--- EXPERIMENT 1 RESULTS ---")
print(f"Training images: {len(X_train)}")
print(f"Test images: {len(X_test)}")
print(f"Test accuracy: {accuracy:.2f}%")
print(f"Training time: {elapsed:.2f} seconds")

# Save this test run's model and metrics in the notebook workspace
torch.save(model.state_dict(), "/content/experiment1_smoke_test.pt")

with open("/content/experiment1_smoke_test.txt", "w") as f:
    f.write(f"Dataset: sklearn Digits (pipeline test only)\n")
    f.write(f"Seed: {SEED}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write(f"Test accuracy: {accuracy:.2f}%\n")
    f.write(f"Training time: {elapsed:.2f} seconds\n")

print("Test model and metrics saved in /content/")


PyTorch: 2.11.0+cpu
Device: cpu
Epoch 1/8 | Loss: 2.1779
Epoch 2/8 | Loss: 1.7923
Epoch 3/8 | Loss: 1.3037
Epoch 4/8 | Loss: 0.9112
Epoch 5/8 | Loss: 0.6630
Epoch 6/8 | Loss: 0.5122
Epoch 7/8 | Loss: 0.4197
Epoch 8/8 | Loss: 0.3555

--- EXPERIMENT 1 RESULTS ---
Training images: 1437
Test images: 360
Test accuracy: 93.33%
Training time: 0.59 seconds
Test model and metrics saved in /content/


In [ ]:

# Research baseline: CIFAR-10, Task A
# CPU-friendly subset; no pretrained model required.

import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print("Device:", device)

# Download CIFAR-10 to Colab's temporary storage
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=True,
    transform=transform
)
test_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=False,
    download=True,
    transform=transform
)

# CIFAR-10 labels:
# 0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer,
# 5 dog, 6 frog, 7 horse, 8 ship, 9 truck
TASK_A = [0, 1, 2, 3, 4]
PER_CLASS_TRAIN = 500
PER_CLASS_TEST = 100

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    chosen = []
    for c in classes:
        indices = np.where(targets == c)[0]
        rng.shuffle(indices)
        chosen.extend(indices[:per_class].tolist())
    return chosen

train_indices = choose_indices(
    train_data, TASK_A, PER_CLASS_TRAIN, SEED
)
test_indices = choose_indices(
    test_data, TASK_A, PER_CLASS_TEST, SEED
)

train_loader = DataLoader(
    Subset(train_data, train_indices),
    batch_size=32,
    shuffle=True,
    num_workers=2
)
test_loader = DataLoader(
    Subset(test_data, test_indices),
    batch_size=64,
    shuffle=False,
    num_workers=2
)

# Small CNN for 32x32 images
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        # Remap original labels 0-4 to task labels 0-4
        labels = labels.to(device)
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_indices):.4f}"
    )

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        predictions = outputs.argmax(dim=1)
        correct += (predictions.cpu() == labels).sum().item()
        total += labels.numel()

accuracy = 100 * correct / total
elapsed = time.time() - start

print("\n--- CIFAR-10 TASK A RESULTS ---")
print("Classes:", TASK_A)
print("Training images:", len(train_indices))
print("Test images:", len(test_indices))
print(f"Test accuracy: {accuracy:.2f}%")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp1_cifar10_taskA.pt")

with open("/content/exp1_cifar10_taskA.txt", "w") as f:
    f.write(f"Seed: {SEED}\n")
    f.write(f"Classes: {TASK_A}\n")
    f.write(f"Train images: {len(train_indices)}\n")
    f.write(f"Test images: {len(test_indices)}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write(f"Accuracy: {accuracy:.2f}%\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Checkpoint and metrics saved.")


Device: cpu
Epoch 1/5 | Loss: 1.3567
Epoch 2/5 | Loss: 1.0545
Epoch 3/5 | Loss: 0.9053
Epoch 4/5 | Loss: 0.8303
Epoch 5/5 | Loss: 0.7600

--- CIFAR-10 TASK A RESULTS ---
Classes: [0, 1, 2, 3, 4]
Training images: 2500
Test images: 500
Test accuracy: 63.40%
Training time: 12.2 seconds
Checkpoint and metrics saved.


In [ ]:

import os
import shutil
import time

print("Checking dataset download...")

dataset_path = "/content/cifar10/cifar-10-batches-py"

if os.path.exists(dataset_path):
    print("Dataset folder exists.")
    print("Files:", os.listdir(dataset_path)[:10])
else:
    print("Dataset folder is not complete or not present.")

total, used, free = shutil.disk_usage("/content")
print(f"Free runtime disk: {free / (1024**3):.2f} GB")


Checking dataset download...
Dataset folder exists.
Files: ['data_batch_1', 'data_batch_4', 'readme.html', 'data_batch_3', 'test_batch', 'data_batch_5', 'data_batch_2', 'batches.meta']
Free runtime disk: 86.84 GB


In [ ]:

# Backup Experiment 1B before sequential training
import os
import shutil

checkpoint = "/content/exp1_cifar10_taskA.pt"
metrics = "/content/exp1_cifar10_taskA.txt"

print("Checkpoint exists:", os.path.exists(checkpoint))
print("Metrics file exists:", os.path.exists(metrics))

if os.path.exists(checkpoint):
    shutil.copy2(
        checkpoint,
        "/content/exp1_cifar10_taskA_backup.pt"
    )
    print("Original model checkpoint backed up.")

if os.path.exists(metrics):
    print("\nBaseline metrics:")
    with open(metrics, "r") as f:
        print(f.read())


Checkpoint exists: True
Metrics file exists: True
Original model checkpoint backed up.

Baseline metrics:
Seed: 42
Classes: [0, 1, 2, 3, 4]
Train images: 2500
Test images: 500
Epochs: 5
Accuracy: 63.40%
Training time: 12.2 seconds



In [ ]:

# EXPERIMENT 2: Sequential learning WITHOUT replay
# Task A -> Task B, then evaluate both tasks.

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cpu")

# Same dataset and preprocessing as Experiment 1B
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=False,
    transform=transform
)
test_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=False,
    download=False,
    transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    chosen = []
    for c in classes:
        indices = np.where(targets == c)[0]
        rng.shuffle(indices)
        chosen.extend(indices[:per_class].tolist())
    return chosen

# Reproduce the same subsets from Experiment 1B
train_A = choose_indices(train_data, TASK_A, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def make_loader(dataset, indices, shuffle):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

loader_A = make_loader(train_data, train_A, True)
loader_B = make_loader(train_data, train_B, True)
test_loader_A = make_loader(test_data, test_A, False)
test_loader_B = make_loader(test_data, test_B, False)

# Identical architecture to Experiment 1B
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            # Map global labels to task-local labels 0-4
            labels = labels - 5 if loader is test_loader_B else labels
            outputs = model(images.to(device))
            predictions = outputs.argmax(dim=1).cpu()
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return 100.0 * correct / total

# Load original Task A model and confirm its baseline score
model = SmallCNN().to(device)
model.load_state_dict(
    torch.load(
        "/content/exp1_cifar10_taskA_backup.pt",
        map_location=device,
        weights_only=True
    )
)

# Check Task A before learning B
accuracy_A_before = evaluate(model, test_loader_A)
print(f"Task A accuracy before Task B: {accuracy_A_before:.2f}%")

# Learn Task B with no old examples in training
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
EPOCHS_B = 5
start = time.time()

for epoch in range(EPOCHS_B):
    model.train()
    total_loss = 0.0

    for images, labels in loader_B:
        # Convert labels 5-9 to 0-4 for the 5-output classifier
        labels = labels - 5
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Task B epoch {epoch + 1}/{EPOCHS_B} | "
        f"Loss: {total_loss / len(train_B):.4f}"
    )

# Evaluate on both tasks after learning B
accuracy_A_after = evaluate(model, test_loader_A)
accuracy_B_after = evaluate(model, test_loader_B)
forgetting = accuracy_A_before - accuracy_A_after

elapsed = time.time() - start

print("\n--- EXPERIMENT 2 RESULTS ---")
print(f"Task A accuracy before B: {accuracy_A_before:.2f}%")
print(f"Task A accuracy after B:  {accuracy_A_after:.2f}%")
print(f"Task B accuracy after B:  {accuracy_B_after:.2f}%")
print(f"Task A accuracy change:   {forgetting:.2f} percentage points")
print(f"Task B training time:    {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp2_sequential_no_replay.pt")

with open("/content/exp2_results.txt", "w") as f:
    f.write(f"Task A before B: {accuracy_A_before:.2f}%\n")
    f.write(f"Task A after B: {accuracy_A_after:.2f}%\n")
    f.write(f"Task B after B: {accuracy_B_after:.2f}%\n")
    f.write(f"Task A accuracy change: {forgetting:.2f} percentage points\n")
    f.write(f"Task B training time: {elapsed:.1f} seconds\n")

print("Experiment 2 checkpoint and results saved.")


Task A accuracy before Task B: 63.40%
Task B epoch 1/5 | Loss: 1.3339
Task B epoch 2/5 | Loss: 0.8772
Task B epoch 3/5 | Loss: 0.7208
Task B epoch 4/5 | Loss: 0.6223
Task B epoch 5/5 | Loss: 0.5215

--- EXPERIMENT 2 RESULTS ---
Task A accuracy before B: 63.40%
Task A accuracy after B:  10.00%
Task B accuracy after B:  74.00%
Task A accuracy change:   53.40 percentage points
Task B training time:    12.7 seconds
Experiment 2 checkpoint and results saved.


In [ ]:

# EXPERIMENT 2 — Corrected sequential learning
# A fixed 10-class output space prevents label remapping.

import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_A = choose_indices(train_data, TASK_A, 500, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def loader(dataset, ids, shuffle=False):
    return DataLoader(
        Subset(dataset, ids),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

loader_A = loader(train_data, train_A, True)
loader_B = loader(train_data, train_B, True)
test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def train_task(model, train_loader, epochs, task_name):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)  # Original labels 0–9

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)

        print(
            f"{task_name} epoch {epoch + 1}/{epochs} | "
            f"Loss: {total_loss / len(train_loader.dataset):.4f}"
        )

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))

            # Evaluate among the classes belonging to this task.
            # This measures task-aware classification, not global
            # 10-class class-incremental accuracy.
            task_scores = outputs[:, allowed_classes]
            predictions = torch.tensor(
                allowed_classes
            )[task_scores.argmax(dim=1)]

            correct += (predictions == labels).sum().item()
            total += labels.numel()

    return 100.0 * correct / total

model = SmallCNN10().to(device)
start = time.time()

# Stage 1: Learn Task A
train_task(model, loader_A, 5, "Task A")
acc_A_before = evaluate_task(model, test_loader_A, TASK_A)
print(f"\nTask A accuracy after Task A training: {acc_A_before:.2f}%")

# Save a checkpoint before learning Task B
torch.save(model.state_dict(), "/content/exp2_corrected_after_A.pt")

# Stage 2: Learn Task B without replaying Task A
train_task(model, loader_B, 5, "Task B")

acc_A_after = evaluate_task(model, test_loader_A, TASK_A)
acc_B_after = evaluate_task(model, test_loader_B, TASK_B)
forgetting = acc_A_before - acc_A_after
elapsed = time.time() - start

print("\n--- EXPERIMENT 2 CORRECTED RESULTS ---")
print(f"Task A before B: {acc_A_before:.2f}%")
print(f"Task A after B:  {acc_A_after:.2f}%")
print(f"Task B after B:  {acc_B_after:.2f}%")
print(f"Task A decline:  {forgetting:.2f} percentage points")
print(f"Total training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp2_corrected_final.pt")

with open("/content/exp2_corrected_results.txt", "w") as f:
    f.write("Fixed 10-output classifier; task-aware evaluation\n")
    f.write(f"Task A before B: {acc_A_before:.2f}%\n")
    f.write(f"Task A after B: {acc_A_after:.2f}%\n")
    f.write(f"Task B after B: {acc_B_after:.2f}%\n")
    f.write(f"Task A decline: {forgetting:.2f} percentage points\n")
    f.write(f"Total training time: {elapsed:.1f} seconds\n")

print("Corrected checkpoint and results saved.")


Task A epoch 1/5 | Loss: 1.3759
Task A epoch 2/5 | Loss: 1.0693
Task A epoch 3/5 | Loss: 0.9320
Task A epoch 4/5 | Loss: 0.8547
Task A epoch 5/5 | Loss: 0.7994

Task A accuracy after Task A training: 65.00%
Task B epoch 1/5 | Loss: 2.6970
Task B epoch 2/5 | Loss: 1.3942
Task B epoch 3/5 | Loss: 1.0488
Task B epoch 4/5 | Loss: 0.9103
Task B epoch 5/5 | Loss: 0.7864

--- EXPERIMENT 2 CORRECTED RESULTS ---
Task A before B: 65.00%
Task A after B:  26.60%
Task B after B:  67.80%
Task A decline:  38.40 percentage points
Total training time: 24.4 seconds
Corrected checkpoint and results saved.


In [ ]:

# EXPERIMENT 3: Experience replay
# Compare against corrected Experiment 2.

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

# Reuse the same dataset and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_A = choose_indices(train_data, TASK_A, 500, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def loader(dataset, indices, shuffle=False):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

# Same architecture as corrected Experiment 2
class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            predictions = outputs[:, allowed_classes].argmax(dim=1)
            predictions = torch.tensor(allowed_classes)[predictions]
            correct += (predictions == labels).sum().item()
            total += labels.numel()

    return 100.0 * correct / total

# Load the same Task A checkpoint used by Experiment 2
checkpoint = "/content/exp2_corrected_after_A.pt"
model = SmallCNN10().to(device)
model.load_state_dict(
    torch.load(checkpoint, map_location=device, weights_only=True)
)

# Deterministic memory: 50 examples per Task A class = 250 examples
memory_indices = choose_indices(train_data, TASK_A, 50, SEED + 1)

# Combine new Task B data with the small old-task memory
combined_dataset = ConcatDataset([
    Subset(train_data, train_B),
    Subset(train_data, memory_indices)
])

combined_loader = DataLoader(
    combined_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

print("Task B training images:", len(train_B))
print("Replay memory images:", len(memory_indices))
print("Combined training images:", len(combined_dataset))

# Evaluate Task A before replay training
acc_A_before = evaluate_task(model, test_loader_A, TASK_A)
print(f"Task A accuracy before B: {acc_A_before:.2f}%")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
EPOCHS = 5
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in combined_loader:
        images = images.to(device)
        labels = labels.to(device)  # Keep original labels 0–9

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    print(
        f"Replay epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(combined_dataset):.4f}"
    )

acc_A_after = evaluate_task(model, test_loader_A, TASK_A)
acc_B_after = evaluate_task(model, test_loader_B, TASK_B)
forgetting = acc_A_before - acc_A_after
elapsed = time.time() - start

print("\n--- EXPERIMENT 3 RESULTS ---")
print(f"Task A before B: {acc_A_before:.2f}%")
print(f"Task A after replay: {acc_A_after:.2f}%")
print(f"Task B after replay: {acc_B_after:.2f}%")
print(f"Task A decline: {forgetting:.2f} percentage points")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp3_replay.pt")

with open("/content/exp3_replay_results.txt", "w") as f:
    f.write("Method: Experience replay\n")
    f.write("Memory size: 250 images\n")
    f.write(f"Task A before B: {acc_A_before:.2f}%\n")
    f.write(f"Task A after replay: {acc_A_after:.2f}%\n")
    f.write(f"Task B after replay: {acc_B_after:.2f}%\n")
    f.write(f"Task A decline: {forgetting:.2f} percentage points\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Experiment 3 checkpoint and results saved.")


Task B training images: 2500
Replay memory images: 250
Combined training images: 2750
Task A accuracy before B: 65.00%
Replay epoch 1/5 | Loss: 2.6238
Replay epoch 2/5 | Loss: 1.6530
Replay epoch 3/5 | Loss: 1.3193
Replay epoch 4/5 | Loss: 1.1228
Replay epoch 5/5 | Loss: 0.9777

--- EXPERIMENT 3 RESULTS ---
Task A before B: 65.00%
Task A after replay: 56.20%
Task B after replay: 71.80%
Task A decline: 8.80 percentage points
Training time: 13.7 seconds
Experiment 3 checkpoint and results saved.


In [ ]:

from google.colab import files
import os

files_to_download = [
    "/content/exp1_cifar10_taskA.txt",
    "/content/exp2_corrected_results.txt",
    "/content/exp3_replay_results.txt",
    "/content/exp2_corrected_after_A.pt",
    "/content/exp2_corrected_final.pt",
    "/content/exp3_replay.pt",
]

for path in files_to_download:
    if os.path.exists(path):
        print("Downloading:", os.path.basename(path))
        files.download(path)
    else:
        print("Not found:", os.path.basename(path))


Downloading: exp1_cifar10_taskA.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: exp2_corrected_results.txt


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: exp3_replay.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# EXPERIMENT 4A: Small replay memory (50 images)
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_A = choose_indices(train_data, TASK_A, 500, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def loader(dataset, indices, shuffle=False):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            scores = model(images.to(device))[:, allowed_classes]
            predictions = torch.tensor(allowed_classes)[scores.argmax(1)]
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return 100 * correct / total

# Start from the same Task A checkpoint
model = SmallCNN10().to(device)
model.load_state_dict(torch.load(
    "/content/exp2_corrected_after_A.pt",
    map_location=device,
    weights_only=True
))

# Exactly 10 memory examples per Task A class = 50
memory_indices = choose_indices(train_data, TASK_A, 10, SEED + 1)

combined = ConcatDataset([
    Subset(train_data, train_B),
    Subset(train_data, memory_indices)
])

train_loader = DataLoader(
    combined, batch_size=32, shuffle=True, num_workers=2
)

before = evaluate_task(model, test_loader_A, TASK_A)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
start = time.time()

for epoch in range(5):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f"Epoch {epoch+1}/5 | Loss: {total_loss/len(combined):.4f}")

after_A = evaluate_task(model, test_loader_A, TASK_A)
after_B = evaluate_task(model, test_loader_B, TASK_B)
elapsed = time.time() - start

print("\n--- EXPERIMENT 4A RESULTS ---")
print(f"Memory size: {len(memory_indices)} images")
print(f"Task A before B: {before:.2f}%")
print(f"Task A after replay: {after_A:.2f}%")
print(f"Task B after replay: {after_B:.2f}%")
print(f"Task A decline: {before-after_A:.2f} percentage points")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp4a_replay_50.pt")

with open("/content/exp4a_results.txt", "w") as f:
    f.write(f"Memory size: {len(memory_indices)}\n")
    f.write(f"Task A before B: {before:.2f}%\n")
    f.write(f"Task A after replay: {after_A:.2f}%\n")
    f.write(f"Task B after replay: {after_B:.2f}%\n")
    f.write(f"Task A decline: {before-after_A:.2f} percentage points\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Experiment 4A results saved.")


Epoch 1/5 | Loss: 2.6265
Epoch 2/5 | Loss: 1.4521
Epoch 3/5 | Loss: 1.2093
Epoch 4/5 | Loss: 1.0254
Epoch 5/5 | Loss: 0.8520

--- EXPERIMENT 4A RESULTS ---
Memory size: 50 images
Task A before B: 65.00%
Task A after replay: 43.60%
Task B after replay: 69.00%
Task A decline: 21.40 percentage points
Training time: 13.1 seconds
Experiment 4A results saved.


In [ ]:

# EXPERIMENT 4B: Replay memory of 500 images
# Run after Experiment 4A.

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

def loader(dataset, indices, shuffle=False):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            scores = model(images.to(device))[:, allowed_classes]
            predictions = torch.tensor(allowed_classes)[scores.argmax(1)]
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return 100 * correct / total

# Load the same checkpoint from after Task A
model = SmallCNN10().to(device)
model.load_state_dict(torch.load(
    "/content/exp2_corrected_after_A.pt",
    map_location=device,
    weights_only=True
))

# 100 images per Task A class = 500 images
memory_indices = choose_indices(train_data, TASK_A, 100, SEED + 1)

combined = ConcatDataset([
    Subset(train_data, train_B),
    Subset(train_data, memory_indices)
])

train_loader = DataLoader(
    combined, batch_size=32, shuffle=True, num_workers=2
)

before = evaluate_task(model, test_loader_A, TASK_A)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
start = time.time()

for epoch in range(5):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    print(
        f"Epoch {epoch+1}/5 | "
        f"Loss: {total_loss / len(combined):.4f}"
    )

after_A = evaluate_task(model, test_loader_A, TASK_A)
after_B = evaluate_task(model, test_loader_B, TASK_B)
elapsed = time.time() - start

print("\n--- EXPERIMENT 4B RESULTS ---")
print("Memory size:", len(memory_indices))
print(f"Task A before B: {before:.2f}%")
print(f"Task A after replay: {after_A:.2f}%")
print(f"Task B after replay: {after_B:.2f}%")
print(f"Task A decline: {before-after_A:.2f} percentage points")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp4b_replay_500.pt")

with open("/content/exp4b_results.txt", "w") as f:
    f.write("Memory size: 500 images\n")
    f.write(f"Task A before B: {before:.2f}%\n")
    f.write(f"Task A after replay: {after_A:.2f}%\n")
    f.write(f"Task B after replay: {after_B:.2f}%\n")
    f.write(f"Task A decline: {before-after_A:.2f} percentage points\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Experiment 4B results saved.")


Epoch 1/5 | Loss: 2.4997
Epoch 2/5 | Loss: 1.7379
Epoch 3/5 | Loss: 1.3994
Epoch 4/5 | Loss: 1.1791
Epoch 5/5 | Loss: 1.0515

--- EXPERIMENT 4B RESULTS ---
Memory size: 500
Task A before B: 65.00%
Task A after replay: 61.40%
Task B after replay: 70.40%
Task A decline: 3.60 percentage points
Training time: 15.6 seconds
Experiment 4B results saved.


In [ ]:

# EXPERIMENT 5: Random replay memory (250 images)
# Compare with Experiment 3's class-balanced 250-image memory.

import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    result = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        result.extend(ids[:per_class].tolist())
    return result

train_A = choose_indices(train_data, TASK_A, 500, SEED)
train_B = choose_indices(train_data, TASK_B, 500, SEED)
test_A = choose_indices(test_data, TASK_A, 100, SEED)
test_B = choose_indices(test_data, TASK_B, 100, SEED)

# Randomly select 250 examples from all 2,500 Task A images
rng = np.random.default_rng(SEED + 10)
memory_indices = rng.choice(
    train_A, size=250, replace=False
).tolist()

print("Random replay memory:", len(memory_indices))
print("Class distribution:", {
    c: int(np.sum(np.array(train_data.targets)[memory_indices] == c))
    for c in TASK_A
})

def loader(dataset, indices, shuffle=False):
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=2
    )

test_loader_A = loader(test_data, test_A)
test_loader_B = loader(test_data, test_B)

class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            scores = model(images.to(device))[:, allowed_classes]
            predictions = torch.tensor(allowed_classes)[scores.argmax(1)]
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return 100.0 * correct / total

# Start from the same Task A checkpoint
model = SmallCNN10().to(device)
model.load_state_dict(torch.load(
    "/content/exp2_corrected_after_A.pt",
    map_location=device,
    weights_only=True
))

combined = ConcatDataset([
    Subset(train_data, train_B),
    Subset(train_data, memory_indices)
])
train_loader = DataLoader(
    combined, batch_size=32, shuffle=True, num_workers=2
)

before = evaluate_task(model, test_loader_A, TASK_A)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
start = time.time()

for epoch in range(5):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    print(
        f"Epoch {epoch+1}/5 | "
        f"Loss: {total_loss / len(combined):.4f}"
    )

after_A = evaluate_task(model, test_loader_A, TASK_A)
after_B = evaluate_task(model, test_loader_B, TASK_B)
elapsed = time.time() - start

print("\n--- EXPERIMENT 5 RESULTS ---")
print("Memory strategy: Random")
print("Memory size:", len(memory_indices))
print(f"Task A before B: {before:.2f}%")
print(f"Task A after replay: {after_A:.2f}%")
print(f"Task B after replay: {after_B:.2f}%")
print(f"Task A decline: {before-after_A:.2f} percentage points")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp5_random_replay.pt")

with open("/content/exp5_random_replay_results.txt", "w") as f:
    f.write("Memory strategy: Random\n")
    f.write(f"Memory size: {len(memory_indices)}\n")
    f.write(f"Task A before B: {before:.2f}%\n")
    f.write(f"Task A after replay: {after_A:.2f}%\n")
    f.write(f"Task B after replay: {after_B:.2f}%\n")
    f.write(f"Task A decline: {before-after_A:.2f} percentage points\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Experiment 5 results saved.")


Random replay memory: 250
Class distribution: {0: 53, 1: 57, 2: 49, 3: 38, 4: 53}
Epoch 1/5 | Loss: 2.6128
Epoch 2/5 | Loss: 1.5520
Epoch 3/5 | Loss: 1.2391
Epoch 4/5 | Loss: 1.0403
Epoch 5/5 | Loss: 0.9162

--- EXPERIMENT 5 RESULTS ---
Memory strategy: Random
Memory size: 250
Task A before B: 65.00%
Task A after replay: 54.00%
Task B after replay: 71.80%
Task A decline: 11.00 percentage points
Training time: 13.5 seconds
Experiment 5 results saved.


In [ ]:

# EXPERIMENT 6: Repeat class-balanced replay with a new training seed
# Fixed data split and memory; different training randomness.

import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

# Change training randomness, but keep dataset and memory selection fixed
DATA_SEED = 42
TRAIN_SEED = 123

random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)

device = torch.device("cpu")
print("Device:", device)
print("Data seed:", DATA_SEED)
print("Training seed:", TRAIN_SEED)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    "/content/cifar10", train=True,
    download=False, transform=transform
)
test_data = datasets.CIFAR10(
    "/content/cifar10", train=False,
    download=False, transform=transform
)

TASK_A = [0, 1, 2, 3, 4]
TASK_B = [5, 6, 7, 8, 9]

def choose_indices(dataset, classes, per_class, seed):
    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    chosen = []
    for c in classes:
        ids = np.where(targets == c)[0]
        rng.shuffle(ids)
        chosen.extend(ids[:per_class].tolist())
    return chosen

# Reproduce exactly the same data subsets as previous experiments
train_A = choose_indices(train_data, TASK_A, 500, DATA_SEED)
train_B = choose_indices(train_data, TASK_B, 500, DATA_SEED)
test_A = choose_indices(test_data, TASK_A, 100, DATA_SEED)
test_B = choose_indices(test_data, TASK_B, 100, DATA_SEED)

# Same class-balanced memory: 50 examples from each Task A class
memory_indices = choose_indices(
    train_data, TASK_A, 50, DATA_SEED + 1
)

def make_loader(dataset, indices, shuffle=False):
    generator = torch.Generator()
    generator.manual_seed(TRAIN_SEED)
    return DataLoader(
        Subset(dataset, indices),
        batch_size=32 if shuffle else 64,
        shuffle=shuffle,
        num_workers=0,
        generator=generator
    )

test_loader_A = make_loader(test_data, test_A)
test_loader_B = make_loader(test_data, test_B)

class SmallCNN10(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate_task(model, test_loader, allowed_classes):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            scores = model(images.to(device))[:, allowed_classes]
            predictions = torch.tensor(allowed_classes)[
                scores.argmax(dim=1)
            ]
            correct += (predictions == labels).sum().item()
            total += labels.numel()

    return 100.0 * correct / total

# Start from the exact same saved Task A checkpoint
checkpoint = "/content/exp2_corrected_after_A.pt"
model = SmallCNN10().to(device)
model.load_state_dict(
    torch.load(checkpoint, map_location=device, weights_only=True)
)

before = evaluate_task(model, test_loader_A, TASK_A)

# Task B + 250 balanced memory examples
combined = ConcatDataset([
    Subset(train_data, train_B),
    Subset(train_data, memory_indices)
])

train_loader = make_loader(combined, list(range(len(combined))), True)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
EPOCHS = 5
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {total_loss / len(combined):.4f}"
    )

after_A = evaluate_task(model, test_loader_A, TASK_A)
after_B = evaluate_task(model, test_loader_B, TASK_B)
decline = before - after_A
elapsed = time.time() - start

print("\n--- EXPERIMENT 6 RESULTS ---")
print("Method: Class-balanced replay")
print("Memory: 250 images (50 per class)")
print("Data seed:", DATA_SEED)
print("Training seed:", TRAIN_SEED)
print(f"Task A before B: {before:.2f}%")
print(f"Task A after replay: {after_A:.2f}%")
print(f"Task B after replay: {after_B:.2f}%")
print(f"Task A decline: {decline:.2f} percentage points")
print(f"Training time: {elapsed:.1f} seconds")

torch.save(model.state_dict(), "/content/exp6_replay_seed123.pt")

with open("/content/exp6_results.txt", "w") as f:
    f.write("Method: Class-balanced replay\n")
    f.write("Memory: 250 images\n")
    f.write(f"Data seed: {DATA_SEED}\n")
    f.write(f"Training seed: {TRAIN_SEED}\n")
    f.write(f"Task A before B: {before:.2f}%\n")
    f.write(f"Task A after replay: {after_A:.2f}%\n")
    f.write(f"Task B after replay: {after_B:.2f}%\n")
    f.write(f"Task A decline: {decline:.2f} percentage points\n")
    f.write(f"Training time: {elapsed:.1f} seconds\n")

print("Experiment 6 results saved.")


Device: cpu
Data seed: 42
Training seed: 123


RuntimeError: Dataset not found or corrupted. You can use download=True to download it

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.CIFAR10(
    root="/content/cifar10",
    train=False,
    download=True,
    transform=transform
)

print("CIFAR-10 downloaded and verified!")
print("Training images:", len(train_data))
print("Test images:", len(test_data))

100%|██████████| 170M/170M [47:25<00:00, 59.9kB/s]


CIFAR-10 downloaded and verified!
Training images: 50000
Test images: 10000


In [ ]:
import os

files_to_check = [
    "/content/exp2_corrected_after_A.pt",
    "/content/exp3_replay.pt",
    "/content/exp4a_replay_50.pt",
    "/content/exp4b_replay_500.pt",
    "/content/exp5_random_replay.pt",
    "/content/exp3_replay_results.txt",
    "/content/exp4a_results.txt",
    "/content/exp4b_results.txt",
    "/content/exp5_random_replay_results.txt"
]

for path in files_to_check:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(status, "—", os.path.basename(path))

MISSING: /content/exp2_corrected_after_A.pt
MISSING: /content/exp3_replay.pt
MISSING: /content/exp4a_replay_50.pt
MISSING: /content/exp4b_replay_500.pt
MISSING: /content/exp5_random_replay.pt
MISSING: /content/exp6_results.txt


In [ ]:
import torch
import torchvision
import random
import numpy as np

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", device)
print("CIFAR-10 training images:", len(train_data))
print("CIFAR-10 test images:", len(test_data))

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
Device: cpu
CIFAR-10 training images: 50000
CIFAR-10 test images: 10000


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Reproducible setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")

# Select Task A: CIFAR-10 classes 0-4
task_a_indices = [
    i for i, label in enumerate(train_data.targets)
    if label in [0, 1, 2, 3, 4]
]

rng = np.random.default_rng(SEED)
selected_indices = []
for class_id in range(5):
    class_indices = [
        i for i in task_a_indices
        if train_data.targets[i] == class_id
    ]
    selected_indices.extend(
        rng.choice(class_indices, size=500, replace=False).tolist()
    )

task_a_train = Subset(train_data, selected_indices)

# Task A test set: 100 images per class
test_indices = []
for class_id in range(5):
    class_indices = [
        i for i, label in enumerate(test_data.targets)
        if label == class_id
    ]
    test_indices.extend(class_indices[:100])

task_a_test = Subset(test_data, test_indices)

train_loader = DataLoader(
    task_a_train, batch_size=64, shuffle=True
)
test_loader = DataLoader(
    task_a_test, batch_size=128, shuffle=False
)

# Small CNN
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train on Task A
for epoch in range(5):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/5 | "
        f"Average loss: {total_loss / len(train_loader):.4f}"
    )

# Task-aware evaluation: predictions restricted to classes 0-4
model.eval()
correct = total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        outputs[:, 5:] = -1e9
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Task A accuracy after training: {accuracy:.2f}%")

# Save checkpoint and essential experiment details
save_dir = "/content/drive/MyDrive/continual_learning"
os.makedirs(save_dir, exist_ok=True)

checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "seed": SEED,
    "task_a_indices": selected_indices,
    "task_a_test_indices": test_indices,
    "task_a_accuracy": accuracy,
    "epochs": 5,
    "classes": [0, 1, 2, 3, 4]
}, checkpoint_path)

print("Checkpoint saved to:", checkpoint_path)

MessageError: Error: credential propagation was unsuccessful

In [12]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [13]:

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

# 1. Reproducible setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")

# 2. Select 500 training images per class for Task A
# Task A = CIFAR-10 classes 0, 1, 2, 3, 4
rng = np.random.default_rng(SEED)
selected_indices = []

for class_id in range(5):
    class_indices = [
        i for i, label in enumerate(train_data.targets)
        if label == class_id
    ]
    chosen = rng.choice(class_indices, size=500, replace=False)
    selected_indices.extend(chosen.tolist())

task_a_train = Subset(train_data, selected_indices)

# 3. Select 100 test images per class
test_indices = [
    i for i, label in enumerate(test_data.targets)
    if label in [0, 1, 2, 3, 4]
]
task_a_test = Subset(test_data, test_indices)

train_loader = DataLoader(
    task_a_train, batch_size=64, shuffle=True
)
test_loader = DataLoader(
    task_a_test, batch_size=128, shuffle=False
)

# 4. Define the CNN
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 5. Train for five epochs
for epoch in range(5):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/5 | "
        f"Average loss: {total_loss / len(train_loader):.4f}"
    )

# 6. Evaluate on Task A's five classes
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        outputs[:, 5:] = -1e9  # Task-aware evaluation
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

task_a_accuracy = 100 * correct / total
print(f"\nTask A accuracy: {task_a_accuracy:.2f}%")

# 7. Save checkpoint to Google Drive
save_dir = "/content/drive/MyDrive/continual_learning"
os.makedirs(save_dir, exist_ok=True)

checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "seed": SEED,
    "task_a_indices": selected_indices,
    "task_a_accuracy": task_a_accuracy,
    "epochs": 5,
    "classes": [0, 1, 2, 3, 4]
}, checkpoint_path)

print("Checkpoint saved:", checkpoint_path)


Epoch 1/5 | Average loss: 1.4002
Epoch 2/5 | Average loss: 1.0498
Epoch 3/5 | Average loss: 0.9119
Epoch 4/5 | Average loss: 0.8346
Epoch 5/5 | Average loss: 0.7509

Task A accuracy: 63.72%
Checkpoint saved: /content/drive/MyDrive/continual_learning/taskA_seed42.pt


In [14]:

import os
import random
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

# -----------------------------
# EXPERIMENT 6: REPEATABILITY
# -----------------------------
TRAIN_SEED = 123

random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)

device = torch.device("cpu")
save_dir = "/content/drive/MyDrive/continual_learning"
checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")

assert os.path.exists(checkpoint_path), (
    "Task A checkpoint not found. Check your Google Drive connection."
)

# Same CNN architecture as the Task A training run
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x))

# Load the identical Task A starting checkpoint
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

model = SmallCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

task_a_indices = checkpoint["task_a_indices"]

# Build fixed, class-balanced memory: 50 images per Task A class
memory_rng = np.random.default_rng(2026)
memory_indices = []

for class_id in range(5):
    candidates = [
        i for i in task_a_indices
        if train_data.targets[i] == class_id
    ]
    chosen = memory_rng.choice(candidates, size=50, replace=False)
    memory_indices.extend(chosen.tolist())

# Fixed Task B subset: 500 images per class, labels 5-9
task_b_rng = np.random.default_rng(43)
task_b_indices = []

for class_id in range(5, 10):
    candidates = [
        i for i, label in enumerate(train_data.targets)
        if label == class_id
    ]
    chosen = task_b_rng.choice(candidates, size=500, replace=False)
    task_b_indices.extend(chosen.tolist())

combined_indices = task_b_indices + memory_indices
combined_data = Subset(train_data, combined_indices)

# Seed controls the training order for this repeat
generator = torch.Generator().manual_seed(TRAIN_SEED)
train_loader = DataLoader(
    combined_data, batch_size=64, shuffle=True, generator=generator
)

# Task-aware evaluation
def evaluate_task(model, dataset, allowed_classes):
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            disallowed = [
                c for c in range(10) if c not in allowed_classes
            ]
            outputs[:, disallowed] = -1e9
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100.0 * correct / total

task_a_test_indices = [
    i for i, label in enumerate(test_data.targets) if label in range(5)
]
task_b_test_indices = [
    i for i, label in enumerate(test_data.targets) if label in range(5, 10)
]
task_a_test = Subset(test_data, task_a_test_indices)
task_b_test = Subset(test_data, task_b_test_indices)

task_a_before = evaluate_task(model, task_a_test, list(range(5)))
print(f"Task A before replay training: {task_a_before:.2f}%")
print("Task B images:", len(task_b_indices))
print("Replay memory images:", len(memory_indices))
print("Training seed:", TRAIN_SEED)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
start_time = time.time()

for epoch in range(5):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/5 | "
        f"Average loss: {total_loss / len(train_loader):.4f}"
    )

training_time = time.time() - start_time

task_a_after = evaluate_task(model, task_a_test, list(range(5)))
task_b_after = evaluate_task(model, task_b_test, list(range(5, 10)))
forgetting = task_a_before - task_a_after

print("\n--- EXPERIMENT 6 RESULTS ---")
print(f"Task A before: {task_a_before:.2f}%")
print(f"Task A after:  {task_a_after:.2f}%")
print(f"Task B after:  {task_b_after:.2f}%")
print(f"Task A decline: {forgetting:.2f} percentage points")
print(f"Training time: {training_time:.2f} seconds")

# Save checkpoint and results
torch.save({
    "model_state_dict": model.state_dict(),
    "train_seed": TRAIN_SEED,
    "memory_indices": memory_indices,
    "task_b_indices": task_b_indices,
    "task_a_before": task_a_before,
    "task_a_after": task_a_after,
    "task_b_after": task_b_after,
    "forgetting_pp": forgetting,
    "training_time_seconds": training_time
}, os.path.join(save_dir, "experiment6_replay_seed123.pt"))

with open(os.path.join(save_dir, "experiment6_results.txt"), "w") as f:
    f.write("Experiment 6: Repeatability test\n")
    f.write(f"Training seed: {TRAIN_SEED}\n")
    f.write(f"Task A before: {task_a_before:.2f}%\n")
    f.write(f"Task A after: {task_a_after:.2f}%\n")
    f.write(f"Task B after: {task_b_after:.2f}%\n")
    f.write(f"Forgetting (pp): {forgetting:.2f}\n")
    f.write(f"Training time (s): {training_time:.2f}\n")

print("Experiment 6 checkpoint and results saved to Google Drive.")


Task A before replay training: 63.72%
Task B images: 2500
Replay memory images: 250
Training seed: 123
Epoch 1/5 | Average loss: 2.6256
Epoch 2/5 | Average loss: 1.6417
Epoch 3/5 | Average loss: 1.2231
Epoch 4/5 | Average loss: 0.9713
Epoch 5/5 | Average loss: 0.8415

--- EXPERIMENT 6 RESULTS ---
Task A before: 63.72%
Task A after:  54.72%
Task B after:  74.76%
Task A decline: 9.00 percentage points
Training time: 27.32 seconds
Experiment 6 checkpoint and results saved to Google Drive.


In [15]:
import os

path = "/content/drive/MyDrive/continual_learning/taskA_seed42.pt"

if os.path.exists(path):
    print("CHECKPOINT FOUND — ready for Experiment 7")
else:
    print("CHECKPOINT NOT FOUND — let's fix it")

CHECKPOINT FOUND — ready for Experiment 7


In [16]:

import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

# Experiment 7: controlled replay comparison
SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

save_dir = "/content/drive/MyDrive/continual_learning"
checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")
assert os.path.exists(checkpoint_path), "Task A checkpoint not found."

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x))

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

# Re-create fixed Task B data: 500 examples per class, labels 5-9
rng = np.random.default_rng(43)
task_b_indices = []
for class_id in range(5, 10):
    candidates = [
        i for i, label in enumerate(train_data.targets)
        if label == class_id
    ]
    task_b_indices.extend(
        rng.choice(candidates, size=500, replace=False).tolist()
    )

task_b_data = Subset(train_data, task_b_indices)

# Fixed balanced replay memory: 50 examples per Task A class
memory_rng = np.random.default_rng(2026)
task_a_indices = checkpoint["task_a_indices"]
memory_indices = []
for class_id in range(5):
    candidates = [
        i for i in task_a_indices
        if train_data.targets[i] == class_id
    ]
    memory_indices.extend(
        memory_rng.choice(candidates, size=50, replace=False).tolist()
    )

task_b_test = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5, 10)]
)
task_a_test = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5)]
)

def evaluate(model, dataset, allowed_classes):
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    model.eval()
    correct = total = 0
    disallowed = [c for c in range(10) if c not in allowed_classes]

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            outputs[:, disallowed] = -1e9
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

def run_condition(use_replay):
    # Reset to the identical Task A weights for each condition
    model = SmallCNN()
    model.load_state_dict(checkpoint["model_state_dict"])

    if use_replay:
        indices = task_b_indices + memory_indices
    else:
        indices = task_b_indices

    data = Subset(train_data, indices)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        data, batch_size=64, shuffle=True, generator=generator
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    start = time.time()

    for epoch in range(5):
        model.train()
        total_loss = 0.0
        for images, labels in loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(
            f"{'Replay' if use_replay else 'No replay'} "
            f"epoch {epoch+1}/5 | loss: {total_loss/len(loader):.4f}"
        )

    elapsed = time.time() - start
    result = {
        "task_a_after": evaluate(model, task_a_test, list(range(5))),
        "task_b_after": evaluate(model, task_b_test, list(range(5, 10))),
        "training_time_seconds": elapsed
    }
    torch.save(
        {"model_state_dict": model.state_dict(), **result},
        os.path.join(
            save_dir,
            "exp7_with_replay.pt" if use_replay else "exp7_without_replay.pt"
        )
    )
    return result

# Run both conditions
no_replay = run_condition(False)
with_replay = run_condition(True)

print("\n--- EXPERIMENT 7 COMPARISON ---")
print("No replay:", no_replay)
print("With 250-image balanced replay:", with_replay)
print(
    "Task A retention difference:",
    round(with_replay["task_a_after"] - no_replay["task_a_after"], 2),
    "percentage points"
)

with open(os.path.join(save_dir, "experiment7_results.txt"), "w") as f:
    f.write(f"No replay: {no_replay}\n")
    f.write(f"With 250-image balanced replay: {with_replay}\n")

print("Experiment 7 results saved to Google Drive.")


No replay epoch 1/5 | loss: 2.7447
No replay epoch 2/5 | loss: 1.4028
No replay epoch 3/5 | loss: 0.9754
No replay epoch 4/5 | loss: 0.7833
No replay epoch 5/5 | loss: 0.7056
Replay epoch 1/5 | loss: 2.6582
Replay epoch 2/5 | loss: 1.7135
Replay epoch 3/5 | loss: 1.2589
Replay epoch 4/5 | loss: 1.0114
Replay epoch 5/5 | loss: 0.8550

--- EXPERIMENT 7 COMPARISON ---
No replay: {'task_a_after': 29.78, 'task_b_after': 73.36, 'training_time_seconds': 25.35543942451477}
With 250-image balanced replay: {'task_a_after': 55.48, 'task_b_after': 72.3, 'training_time_seconds': 25.633827686309814}
Task A retention difference: 25.7 percentage points
Experiment 7 results saved to Google Drive.


In [17]:

import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

# Experiment 8: controlled memory-sampling comparison
SEED = 888
save_dir = "/content/drive/MyDrive/continual_learning"
checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")

assert os.path.exists(checkpoint_path), "Task A checkpoint not found."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x))

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)
task_a_indices = checkpoint["task_a_indices"]

# Fixed Task B split: 500 images per class, classes 5-9
rng_b = np.random.default_rng(43)
task_b_indices = []
for class_id in range(5, 10):
    candidates = [
        i for i, label in enumerate(train_data.targets)
        if label == class_id
    ]
    task_b_indices.extend(
        rng_b.choice(candidates, size=500, replace=False).tolist()
    )

# Fixed class-balanced memory: 50 examples per Task A class
rng_balanced = np.random.default_rng(2026)
balanced_indices = []
for class_id in range(5):
    candidates = [
        i for i in task_a_indices
        if train_data.targets[i] == class_id
    ]
    balanced_indices.extend(
        rng_balanced.choice(candidates, size=50, replace=False).tolist()
    )

# Fixed random memory: 250 images sampled from Task A
rng_random = np.random.default_rng(2027)
random_indices = rng_random.choice(
    task_a_indices, size=250, replace=False
).tolist()

task_a_test = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5)]
)
task_b_test = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5, 10)]
)

def evaluate(model, dataset, allowed_classes):
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    model.eval()
    correct = total = 0
    disallowed = [c for c in range(10) if c not in allowed_classes]

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            outputs[:, disallowed] = -1e9
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += len(labels)

    return 100.0 * correct / total

def run_memory_condition(memory_indices, label, order_seed):
    # Restore the same initial Task A model
    model = SmallCNN()
    model.load_state_dict(checkpoint["model_state_dict"])

    combined_indices = task_b_indices + memory_indices
    loader = DataLoader(
        Subset(train_data, combined_indices),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(order_seed)
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    start = time.time()

    for epoch in range(5):
        model.train()
        total_loss = 0.0

        for images, labels in loader:
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(
            f"{label} epoch {epoch+1}/5 | "
            f"loss: {total_loss/len(loader):.4f}"
        )

    elapsed = time.time() - start
    result = {
        "task_a_after": evaluate(model, task_a_test, list(range(5))),
        "task_b_after": evaluate(model, task_b_test, list(range(5, 10))),
        "training_time_seconds": elapsed,
        "memory_size": len(memory_indices)
    }

    safe_label = label.lower().replace(" ", "_")
    torch.save(
        {"model_state_dict": model.state_dict(), **result},
        os.path.join(save_dir, f"exp8_{safe_label}.pt")
    )
    return result

balanced_result = run_memory_condition(
    balanced_indices, "Balanced replay", SEED
)
random_result = run_memory_condition(
    random_indices, "Random replay", SEED
)

print("\n--- EXPERIMENT 8 RESULTS ---")
print("Balanced replay:", balanced_result)
print("Random replay:", random_result)
print(
    "Task A retention difference (balanced - random):",
    round(
        balanced_result["task_a_after"] - random_result["task_a_after"],
        2
    ),
    "percentage points"
)

with open(os.path.join(save_dir, "experiment8_results.txt"), "w") as f:
    f.write(f"Balanced replay: {balanced_result}\n")
    f.write(f"Random replay: {random_result}\n")

print("Experiment 8 results saved to Google Drive.")


Balanced replay epoch 1/5 | loss: 2.6514
Balanced replay epoch 2/5 | loss: 1.6930
Balanced replay epoch 3/5 | loss: 1.1988
Balanced replay epoch 4/5 | loss: 0.9759
Balanced replay epoch 5/5 | loss: 0.8197
Random replay epoch 1/5 | loss: 2.6552
Random replay epoch 2/5 | loss: 1.6772
Random replay epoch 3/5 | loss: 1.1851
Random replay epoch 4/5 | loss: 0.9670
Random replay epoch 5/5 | loss: 0.7947

--- EXPERIMENT 8 RESULTS ---
Balanced replay: {'task_a_after': 57.42, 'task_b_after': 74.24, 'training_time_seconds': 27.371530294418335, 'memory_size': 250}
Random replay: {'task_a_after': 59.84, 'task_b_after': 73.7, 'training_time_seconds': 25.757927179336548, 'memory_size': 250}
Task A retention difference (balanced - random): -2.42 percentage points
Experiment 8 results saved to Google Drive.


In [18]:

import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

SEED = 999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

save_dir = "/content/drive/MyDrive/continual_learning"
checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")
assert os.path.exists(checkpoint_path), "Task A checkpoint not found."

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x))

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

# Frozen teacher: model trained on Task A
teacher = SmallCNN()
teacher.load_state_dict(checkpoint["model_state_dict"])
teacher.eval()
for parameter in teacher.parameters():
    parameter.requires_grad = False

# Student begins with exactly the same Task A checkpoint
student = SmallCNN()
student.load_state_dict(checkpoint["model_state_dict"])

# Fixed Task B training split: 500 images per class, classes 5-9
rng_b = np.random.default_rng(43)
task_b_indices = []
for class_id in range(5, 10):
    candidates = [
        i for i, label in enumerate(train_data.targets)
        if label == class_id
    ]
    task_b_indices.extend(
        rng_b.choice(candidates, size=500, replace=False).tolist()
    )

# Fixed class-balanced Task A memory: 50 images per class
rng_mem = np.random.default_rng(2026)
task_a_indices = checkpoint["task_a_indices"]
memory_indices = []
for class_id in range(5):
    candidates = [
        i for i in task_a_indices
        if train_data.targets[i] == class_id
    ]
    memory_indices.extend(
        rng_mem.choice(candidates, size=50, replace=False).tolist()
    )

# Training dataset contains Task B plus the old-task memory
combined_indices = task_b_indices + memory_indices
loader = DataLoader(
    Subset(train_data, combined_indices),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

# Evaluation sets
task_a_test = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5)]
)
task_b_test = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5, 10)]
)

def evaluate(model, dataset, allowed_classes):
    eval_loader = DataLoader(dataset, batch_size=128, shuffle=False)
    model.eval()
    correct = total = 0
    disallowed = [c for c in range(10) if c not in allowed_classes]

    with torch.no_grad():
        for images, labels in eval_loader:
            logits = model(images)
            logits[:, disallowed] = -1e9
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += len(labels)

    return 100.0 * correct / total

task_a_before = evaluate(student, task_a_test, list(range(5)))

# Knowledge distillation settings
temperature = 2.0
alpha = 0.5  # weight on distillation loss
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(student.parameters(), lr=0.001)

start_time = time.time()

for epoch in range(5):
    student.train()
    total_loss = 0.0

    for images, labels in loader:
        optimizer.zero_grad()

        student_logits = student(images)

        # Standard supervised loss on true labels
        classification_loss = criterion(student_logits, labels)

        # Preserve teacher outputs on the old Task A classes.
        # Distillation is applied to every example, but only the old-class
        # logits are used in this preservation term.
        with torch.no_grad():
            teacher_logits = teacher(images)

        student_old = student_logits[:, :5] / temperature
        teacher_old = teacher_logits[:, :5] / temperature

        distillation_loss = F.kl_div(
            F.log_softmax(student_old, dim=1),
            F.softmax(teacher_old, dim=1),
            reduction="batchmean"
        ) * (temperature ** 2)

        loss = (1 - alpha) * classification_loss + alpha * distillation_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Distillation epoch {epoch + 1}/5 | "
        f"Average loss: {total_loss / len(loader):.4f}"
    )

training_time = time.time() - start_time

task_a_after = evaluate(student, task_a_test, list(range(5)))
task_b_after = evaluate(student, task_b_test, list(range(5, 10)))
forgetting = task_a_before - task_a_after

print("\n--- EXPERIMENT 9 RESULTS ---")
print(f"Task A before: {task_a_before:.2f}%")
print(f"Task A after:  {task_a_after:.2f}%")
print(f"Task B after:  {task_b_after:.2f}%")
print(f"Task A decline: {forgetting:.2f} percentage points")
print(f"Training time: {training_time:.2f} seconds")

torch.save({
    "model_state_dict": student.state_dict(),
    "task_a_before": task_a_before,
    "task_a_after": task_a_after,
    "task_b_after": task_b_after,
    "forgetting_pp": forgetting,
    "method": "knowledge_distillation",
    "temperature": temperature,
    "alpha": alpha,
    "seed": SEED
}, os.path.join(save_dir, "experiment9_distillation.pt"))

with open(os.path.join(save_dir, "experiment9_results.txt"), "w") as f:
    f.write("Method: knowledge distillation\n")
    f.write(f"Task A before: {task_a_before:.2f}%\n")
    f.write(f"Task A after: {task_a_after:.2f}%\n")
    f.write(f"Task B after: {task_b_after:.2f}%\n")
    f.write(f"Forgetting (pp): {forgetting:.2f}\n")
    f.write(f"Training time (s): {training_time:.2f}\n")

print("Experiment 9 results saved to Google Drive.")


Distillation epoch 1/5 | Average loss: 1.7034
Distillation epoch 2/5 | Average loss: 0.7435
Distillation epoch 3/5 | Average loss: 0.5291
Distillation epoch 4/5 | Average loss: 0.4460
Distillation epoch 5/5 | Average loss: 0.3955

--- EXPERIMENT 9 RESULTS ---
Task A before: 63.72%
Task A after:  63.72%
Task B after:  74.94%
Task A decline: 0.00 percentage points
Training time: 35.15 seconds
Experiment 9 results saved to Google Drive.


In [19]:

# EXPERIMENT 10
# Repeat Experiment 9 with a different training seed

import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

save_dir = "/content/drive/MyDrive/continual_learning"
checkpoint_path = os.path.join(save_dir, "taskA_seed42.pt")

assert os.path.exists(checkpoint_path), "Checkpoint missing"
assert "train_data" in globals(), "Run the CIFAR-10 loading cell first"
assert "test_data" in globals(), "Run the CIFAR-10 loading cell first"
assert "SmallCNN" in globals(), "Run the Experiment 9 model definition first"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Rebuild the exact same Task B subset
rng_b = np.random.default_rng(43)
task_b_indices = []

for class_id in range(5, 10):
    candidates = [
        i for i, label in enumerate(train_data.targets)
        if label == class_id
    ]
    task_b_indices.extend(
        rng_b.choice(candidates, size=500, replace=False).tolist()
    )

# Rebuild the same balanced replay memory
rng_mem = np.random.default_rng(2026)
memory_indices = []

for class_id in range(5):
    candidates = [
        i for i in checkpoint["task_a_indices"]
        if train_data.targets[i] == class_id
    ]
    memory_indices.extend(
        rng_mem.choice(candidates, size=50, replace=False).tolist()
    )

# Teacher model
teacher10 = SmallCNN()
teacher10.load_state_dict(checkpoint["model_state_dict"])
teacher10.eval()

for p in teacher10.parameters():
    p.requires_grad = False

# Student model
student10 = SmallCNN()
student10.load_state_dict(checkpoint["model_state_dict"])

# Training data
combined_indices = task_b_indices + memory_indices

loader10 = DataLoader(
    Subset(train_data, combined_indices),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

# Evaluation data
task_a_test10 = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5)]
)

task_b_test10 = Subset(
    test_data,
    [i for i, y in enumerate(test_data.targets) if y in range(5, 10)]
)

def evaluate10(model, dataset, allowed_classes):
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    model.eval()

    correct = 0
    total = 0
    disallowed = [
        c for c in range(10) if c not in allowed_classes
    ]

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            outputs[:, disallowed] = -1e9
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total

task_a_before = evaluate10(
    student10, task_a_test10, list(range(5))
)

optimizer = torch.optim.Adam(
    student10.parameters(), lr=0.001
)

criterion = nn.CrossEntropyLoss()

temperature = 2.0
alpha = 0.5

start_time = time.time()

for epoch in range(5):
    student10.train()
    total_loss = 0.0

    for images, labels in loader10:
        optimizer.zero_grad()

        student_logits = student10(images)

        classification_loss = criterion(
            student_logits, labels
        )

        with torch.no_grad():
            teacher_logits = teacher10(images)

        student_old = student_logits[:, :5] / temperature
        teacher_old = teacher_logits[:, :5] / temperature

        distillation_loss = F.kl_div(
            F.log_softmax(student_old, dim=1),
            F.softmax(teacher_old, dim=1),
            reduction="batchmean"
        ) * (temperature ** 2)

        loss = (
            (1 - alpha) * classification_loss
            + alpha * distillation_loss
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/5 | "
        f"Average loss: {total_loss / len(loader10):.4f}"
    )

training_time = time.time() - start_time

task_a_after = evaluate10(
    student10, task_a_test10, list(range(5))
)

task_b_after = evaluate10(
    student10, task_b_test10, list(range(5, 10))
)

forgetting = task_a_before - task_a_after

print("\n--- EXPERIMENT 10 RESULTS ---")
print(f"Task A before: {task_a_before:.2f}%")
print(f"Task A after: {task_a_after:.2f}%")
print(f"Task B after: {task_b_after:.2f}%")
print(f"Task A decline: {forgetting:.2f} percentage points")
print(f"Training time: {training_time:.2f} seconds")

# Save all results to Google Drive
os.makedirs(save_dir, exist_ok=True)

torch.save({
    "model_state_dict": student10.state_dict(),
    "seed": SEED,
    "task_a_before": task_a_before,
    "task_a_after": task_a_after,
    "task_b_after": task_b_after,
    "forgetting_pp": forgetting,
    "training_time_seconds": training_time,
    "method": "replay_plus_knowledge_distillation",
    "temperature": temperature,
    "alpha": alpha,
    "memory_indices": memory_indices,
    "task_b_indices": task_b_indices
}, os.path.join(save_dir, "experiment10_distillation.pt"))

with open(
    os.path.join(save_dir, "experiment10_results.txt"), "w"
) as f:
    f.write("Experiment 10: Distillation repeatability\n")
    f.write(f"Seed: {SEED}\n")
    f.write(f"Task A before: {task_a_before:.2f}%\n")
    f.write(f"Task A after: {task_a_after:.2f}%\n")
    f.write(f"Task B after: {task_b_after:.2f}%\n")
    f.write(f"Forgetting (pp): {forgetting:.2f}\n")
    f.write(f"Training time (s): {training_time:.2f}\n")

print("Experiment 10 results saved to Google Drive!")


Epoch 1/5 | Average loss: 1.6994
Epoch 2/5 | Average loss: 0.7406
Epoch 3/5 | Average loss: 0.5243
Epoch 4/5 | Average loss: 0.4527
Epoch 5/5 | Average loss: 0.3882

--- EXPERIMENT 10 RESULTS ---
Task A before: 63.72%
Task A after: 63.98%
Task B after: 75.64%
Task A decline: -0.26 percentage points
Training time: 36.10 seconds
Experiment 10 results saved to Google Drive!
